# LAB 11 - Human–AI Judgment Bias
### Decision Tree · Random Forest · XGBoost

**Disciplina:** Inteligência Artificial - PPGI/UNIRIO (2026.2)  
**Base:** `Exp2.csv` - Glickman & Sharot (Nature Human Behaviour, 2025)

**Questão:** o histórico de julgamentos e interações anteriores permite prever melhor os desvios das novas estimativas humanas?

**Nota crítica:** prever um desvio numérico não comprova viés cognitivo individual nem mecanismo causal.

## 1. Ambiente e importação

Este notebook pode ser aberto tanto da pasta raiz quanto da pasta `notebooks/`. O CSV foi incluído no pacote, mas é ignorado no Git por padrão. Instale `requirements.txt` antes de executar.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'data' / 'Exp2.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'Exp2.csv').exists(), 'Abra o notebook no laboratório ou copie data/Exp2.csv.'
sys.path.insert(0, str(ROOT))

from src.data import (load_data, audit_data, make_features, STATIC_FEATURES,
                      HISTORY_FEATURES, CONDITIONS, TARGET)
from src.models import participant_split, features_for, make_pipeline, scores

print('Raiz:', ROOT)

## 2. Auditoria do CSV

A unidade de observação é uma **tentativa**, não uma pessoa independente. `collaboration` é ausente nas 30 tentativas baseline por participante, o que decorre do desenho experimental.

In [ ]:
raw = load_data(ROOT / 'data' / 'Exp2.csv')
audit = audit_data(raw, strict=True)
print(json.dumps(audit, ensure_ascii=False, indent=2))
display(raw.head(5))
display(raw.isna().sum().to_frame('Valores ausentes'))

## 3. Exploração das quatro condições

Os códigos são: `0` sem IA, `1` IA precisa, `2` IA enviesada, `3` IA ruidosa. Abaixo, o **erro assinado** positivo é uma sobrestimação da proporção de pontos; o erro absoluto mede distância da evidência real.

In [ ]:
frame = make_features(raw)
by_condition = frame.groupby('condition').agg(
    registros=('subject', 'size'), participantes=('subject', 'nunique'),
    desvio_medio=(TARGET, 'mean'), mae_humano=('absolute_error', 'mean'))
by_condition.index = [CONDITIONS[c] for c in by_condition.index]
display(by_condition.round(3))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_condition.index, by_condition['desvio_medio'])
ax.axhline(0, color='0.3', linewidth=1)
ax.set_ylabel('Desvio assinado médio (p.p.)')
ax.set_title('Estimativas humanas por condição')
plt.tight_layout();plt.show()

## 4. Tempo experimental e vazamento de informação

**Sequência da tentativa t:** estímulo → estimativa humana (`response_t`) → confiança → ponderação → **revelação** de `responseAI_t`. Assim, a resposta da IA na tentativa *t* não pode explicar diretamente a estimativa humana **anterior** na mesma tentativa.

As features históricas são construídas com `shift(1)` **por participante**. O uso do histórico observado de uma nova pessoa antes do instante *t* configura previsão **online**, não *cold start* estrito.

In [ ]:
participant = frame[frame.subject == 1]
show = ['trial_index','condition','evidence','response','responseAI',
        'shown_ai_bias','signed_error','lag1_error','lag1_shown_ai_bias']
display(pd.concat([participant.iloc[28:33], participant.iloc[58:63]])[show])
print('Contexto:', STATIC_FEATURES)
print('Histórico adicional:', HISTORY_FEATURES)
assert set(features_for('historico')).isdisjoint(
    {'response', 'responseAI', 'confidence', 'collaboration', 'subject'})

## 5. Divisão por participante (group holdout)

O mesmo participante **nunca** aparece simultaneamente em treino e teste. A validação cruzada completa ocorre no script principal, também agrupada por pessoa. Para estudo didático do notebook, abaixo construímos uma versão comparativa de RF com os mesmos hiperparâmetros.

In [ ]:
train_idx, test_idx = participant_split(frame, seed=42, test_size=.25)
train, test = frame.iloc[train_idx].copy(), frame.iloc[test_idx].copy()
print('Pessoas no treino:', train.subject.nunique())
print('Pessoas no teste:', test.subject.nunique())
print('Interseção:', set(train.subject) & set(test.subject))
assert set(train.subject).isdisjoint(test.subject)

## 6. Baseline e comparação didática de Random Forest

O baseline prevê que o participante acertará a proporção do estímulo (desvio **zero**). Treinamos RF no conjunto de treino sob duas condições: `contexto` versus `historico`. Isso demonstra a ablação principal sem repetir todo o pipeline de avaliação cruzada do script.

In [ ]:
predictions = {'Zero': np.zeros(len(test))}
for scenario in ['contexto','historico']:
    columns = features_for(scenario)
    model = make_pipeline('Random Forest', columns, seed=42, quick=True)
    model.fit(train[columns], train[TARGET])
    predictions[f'RF_{scenario}'] = model.predict(test[columns])
for name, values in predictions.items():
    print(name, {k: round(v,3) for k,v in scores(test[TARGET],values).items()})

## 7. Execução completa: DT, RF e XGBoost

Os resultados completos são gerados pelo terminal: `python run_lab.py`. Caso ainda não estejam disponíveis, esta célula os produz usando a configuração **completa**, com validação cruzada agrupada nos participantes do treino.

In [ ]:
results_dir = ROOT / 'results'
metrics_path = results_dir / 'METRICAS_MODELOS.csv'
if not metrics_path.exists():
    from src.experiment import execute
    execute(ROOT / 'data' / 'Exp2.csv', results_dir, quick=False, cv=True, seed=42)
metrics = pd.read_csv(metrics_path)
show = ['tipo','modelo','cenario','MAE','RMSE','R2','cv_mae_mean','late_block_MAE']
display(metrics[show].round(3))

## 8. Desempenho por condição

Os efeitos preditivos podem variar entre contexto sem IA, IA precisa, enviesada e ruidosa. A tabela abaixo avalia os mesmos **participantes não vistos** do conjunto de teste, mas separa suas tentativas por condição.

In [ ]:
condition_scores = pd.read_csv(results_dir / 'METRICAS_POR_CONDICAO.csv')
selection = condition_scores[
    (condition_scores.modelo.isin(['Random Forest','XGBoost'])) &
    (condition_scores.cenario.isin(['contexto','historico']))
]
selection = selection.pivot_table(index=['modelo','condition'], columns='cenario',values='MAE')
selection['ganho'] = selection['contexto'] - selection['historico']
display(selection.round(3))

## 9. Incerteza de comparação por pessoa

O intervalo foi calculado reamostrando **participantes inteiros**, e não tentando tratar 3.600 linhas do teste como eventos independentes. Isso descreve a incerteza da diferença de MAE **para estes modelos treinados**.

In [ ]:
effects = pd.read_csv(results_dir / 'INCERTEZA_HISTORICO.csv')
display(effects.round(3))
fig,ax=plt.subplots(figsize=(8,4))
ax.errorbar(effects.delta_mae, effects.modelo,
            xerr=[effects.delta_mae-effects.ci95_low,
                  effects.ci95_high-effects.delta_mae],
            fmt='o',capsize=5)
ax.axvline(0,linestyle='--',color='0.5')
ax.set_xlabel('Δ MAE = MAE sem histórico − MAE com histórico')
ax.set_title('Ablação de histórico: ganho e IC95% por participantes')
plt.tight_layout();plt.show()

## 10. Gráficos da execução principal

In [ ]:
from IPython.display import Image
for fn in ['01_vies_por_condicao.png','02_mae_modelos.png','04_real_vs_previsto.png']:
    display(Markdown(f'**{fn}**'))
    display(Image(filename=str(results_dir/fn)))

## 11. Discussão crítica (responda com suas palavras)

1. Por que não é permitido usar `responseAI_t` para prever `response_t` no mesmo trial?
2. Como a validação por participante difere de uma divisão aleatória de linhas?
3. O histórico melhora a predição porque a IA causa viés ou porque os julgamentos são temporalmente autocorrelacionados? Que novo experimento separaria as explicações?
4. Qual é a diferença entre `response - evidence` e uma demonstração de viés sistemático?
5. Por que Random Forest poderia superar XGBoost em MAE, mas perder em R²?
6. O que as limitações de importância de atributos apontadas por Strobl et al. sugerem sobre interpretações de nossos modelos?

**Extensão sugerida:** adicionar um terceiro cenário usando apenas **histórico de conselho revelado** e comparar com um quarto usando apenas **histórico comportamental humano**, sob a mesma divisão e métricas.

## Referências

- Glickman & Sharot (2025), *Nature Human Behaviour*: https://doi.org/10.1038/s41562-024-02077-2.
- Strobl et al. (2007), *BMC Bioinformatics*.
- Chen & Guestrin (2016), *XGBoost: A Scalable Tree Boosting System*.

**Reprodutibilidade:** use `python run_lab.py` e `python -m pytest -v` para validar o pacote completo.